# Asignación de límites regionales de actividad industrial

Distribuye el `TotalTechnologyAnnualActivityLowerLimit` nacional entre las 7 regiones, multiplicando el valor nacional por la matriz de participaciones porcentuales previamente calculada, y exporta el resultado en el formato del SAND Regional.

In [ ]:
import re
import pandas as pd

pd.set_option('display.max_rows', 200)

In [ ]:
# --- Configuración ---
NACIONAL_PATH = "SAND_Nacional_base/01-04-2026 SAND BASE v10.xlsx"
PORCENTAJES_PATH = "Insumos/Participacion_Regional_Industrial.xlsx"
PORCENTAJES_SHEET = "Participacion_Plano"
OUTPUT_PATH = "SAND_Regional/scenario_23_Parameters_SAND_LowerLimit_Industria.xlsx"

PARAMETRO = "TotalTechnologyAnnualActivityLowerLimit"
REGION_OSEMOSYS = "RE1"

# Prefijo de región OSeMOSYS <- nombre de región usado en el cálculo de participaciones
PREFIJO_REGION = {
    "Antioquia": "AN",
    "Caribe": "CA",
    "Este": "SE",
    "Insular": "IN",
    "Nordeste": "NE",
    "Oriente": "OR",
    "Suroccidente": "SO",
}

In [ ]:
# --- Diccionario de mapeo: código OSeMOSYS -> ruta de texto ---
MAPEO_TECNOLOGIA = {
    "DEMINDBAGBOI_LOW": "Calor indirecto\\Bagazo\\Eficiencia_existente",
    "DEMINDBAGBOI_MID": "Calor indirecto\\Bagazo\\Mejor eficiencia_Colombia",
    "DEMINDBAGFUR_LOW": "Calor directo\\Bagazo\\Eficiencia_existente",
    "DEMINDBAGFUR_MID": "Calor directo\\Bagazo\\Mejor eficiencia_Colombia",
    "DEMINDCOABOI_LOW": "Calor indirecto\\Carbón mineral\\Eficiencia_existente",
    "DEMINDCOABOI_MID": "Calor indirecto\\Carbón mineral\\Mejor eficiencia_Colombia",
    "DEMINDCOAFUR_LOW": "Calor directo\\Carbón mineral\\Eficiencia_existente",
    "DEMINDCOAFUR_MID": "Calor directo\\Carbón mineral\\Mejor eficiencia_Colombia",
    "DEMINDCOAOTH_LOW": "Otros\\Carbón mineral\\Eficiencia_existente",
    "DEMINDDSLBOI_LOW": "Calor indirecto\\Diesel\\Eficiencia_existente",
    "DEMINDDSLBOI_MID": "Calor indirecto\\Diesel\\Mejor eficiencia_Colombia",
    "DEMINDDSLFUR_LOW": "Calor directo\\Diesel\\Eficiencia_existente",
    "DEMINDDSLFUR_MID": "Calor directo\\Diesel\\Mejor eficiencia_Colombia",
    "DEMINDELCAIR_LOW": "Aire acondicionado\\Electricidad\\Eficiencia_existente",
    "DEMINDELCAIR_MID": "Aire acondicionado\\Electricidad\\Mejor eficiencia_Colombia",
    "DEMINDELCBOI_LOW": "Calor indirecto\\Electricidad\\Eficiencia_existente",
    "DEMINDELCBOI_MID": "Calor indirecto\\Electricidad\\Mejor eficiencia_Colombia",
    "DEMINDELCFUR_LOW": "Calor directo\\Electricidad\\Eficiencia_existente",
    "DEMINDELCFUR_MID": "Calor directo\\Electricidad\\Mejor eficiencia_Colombia",
    "DEMINDELCILU_LOW": "Iluminacion\\Electricidad\\Eficiencia_existente",
    "DEMINDELCILU_MID": "Iluminacion\\Electricidad\\Mejor eficiencia_Colombia",
    "DEMINDELCMPW_LOW": "Fuerza motriz\\Electricidad\\Eficiencia_existente",
    "DEMINDELCMPW_MID": "Fuerza motriz\\Electricidad\\Mejor eficiencia_Colombia",
    "DEMINDELCOTH_LOW": "Otros\\Electricidad\\Eficiencia_existente",
    "DEMINDELCOTH_MID": "Otros\\Electricidad\\Mejor eficiencia_Colombia",
    "DEMINDELCREF_LOW": "Refrigeracion\\Electricidad\\Eficiencia_existente",
    "DEMINDELCREF_MID": "Refrigeracion\\Electricidad\\Mejor eficiencia_Colombia",
    "DEMINDLPGBOI_LOW": "Calor indirecto\\GLP\\Eficiencia_existente",
    "DEMINDLPGBOI_MID": "Calor indirecto\\GLP\\Mejor eficiencia_Colombia",
    "DEMINDLPGFUR_LOW": "Calor directo\\GLP\\Eficiencia_existente",
    "DEMINDLPGFUR_MID": "Calor directo\\GLP\\Mejor eficiencia_Colombia",
    "DEMINDNGSBOI_LOW": "Calor indirecto\\Gas Natural\\Eficiencia_existente",
    "DEMINDNGSBOI_MID": "Calor indirecto\\Gas Natural\\Mejor eficiencia_Colombia",
    "DEMINDNGSFUR_LOW": "Calor directo\\Gas Natural\\Eficiencia_existente",
    "DEMINDNGSFUR_MID": "Calor directo\\Gas Natural\\Mejor eficiencia_Colombia",
    "DEMINDWASBOI_LOW": "Calor indirecto\\Resíduos\\Eficiencia_existente",
    "DEMINDWASBOI_MID": "Calor indirecto\\Resíduos\\Mejor eficiencia_Colombia",
    "DEMINDWASFUR_LOW": "Calor directo\\Resíduos\\Eficiencia_existente",
    "DEMINDWASFUR_MID": "Calor directo\\Resíduos\\Mejor eficiencia_Colombia",
}
print(f"{len(MAPEO_TECNOLOGIA)} códigos mapeados")

## 1. Cargar y filtrar el Dataframe Nacional

In [ ]:
df_nacional_raw = pd.read_excel(NACIONAL_PATH, sheet_name="Parameters")

df_nac_filt = df_nacional_raw[
    (df_nacional_raw["Parameter"] == PARAMETRO)
    & (df_nacional_raw["TECHNOLOGY"].isin(MAPEO_TECNOLOGIA.keys()))
].copy()

print(f"Filas nacionales encontradas: {len(df_nac_filt)}")
print("Códigos encontrados:", sorted(df_nac_filt["TECHNOLOGY"].unique()))

codigos_sin_dato = set(MAPEO_TECNOLOGIA.keys()) - set(df_nac_filt["TECHNOLOGY"].unique())
if codigos_sin_dato:
    print(f"ADVERTENCIA: códigos del diccionario sin fila en el Nacional: {sorted(codigos_sin_dato)}")

## 2. Formato largo del Nacional

In [ ]:
YEAR_COLS = [c for c in df_nacional_raw.columns if re.fullmatch(r"\d{4}", str(c))]

df_nac_long = df_nac_filt[["TECHNOLOGY"] + YEAR_COLS].melt(
    id_vars="TECHNOLOGY", value_vars=YEAR_COLS, var_name="Año", value_name="Valor_Nacional"
)
df_nac_long = df_nac_long.rename(columns={"TECHNOLOGY": "Codigo_Tecnologia"})
df_nac_long["Año"] = df_nac_long["Año"].astype(int)
df_nac_long["Valor_Nacional"] = pd.to_numeric(df_nac_long["Valor_Nacional"], errors="coerce").fillna(0)

# Texto de ruta correspondiente, usado como llave de cruce con las participaciones
df_nac_long["Tecnologia_Texto"] = df_nac_long["Codigo_Tecnologia"].map(MAPEO_TECNOLOGIA)

# Llave normalizada (minúsculas) para el cruce: el archivo de participaciones no siempre
# respeta exactamente las mayúsculas/minúsculas del diccionario (ej. "Gas Natural" vs "Gas natural")
df_nac_long["Tecnologia_Texto_Norm"] = df_nac_long["Tecnologia_Texto"].str.lower().str.strip()

df_nac_long.head()

## 3. Cargar el Dataframe de Porcentajes

In [ ]:
df_porcentajes = pd.read_excel(PORCENTAJES_PATH, sheet_name=PORCENTAJES_SHEET)
df_porcentajes = df_porcentajes.rename(
    columns={"Tecnologia": "Tecnologia_Texto", "Anio": "Año", "Participacion": "Porcentaje"}
)[["Region", "Tecnologia_Texto", "Año", "Porcentaje"]]

df_porcentajes["Porcentaje"] = pd.to_numeric(df_porcentajes["Porcentaje"], errors="coerce")
df_porcentajes["Tecnologia_Texto_Norm"] = df_porcentajes["Tecnologia_Texto"].str.lower().str.strip()

# --- Relleno de años faltantes con el porcentaje del año anterior ---
# Grilla completa Region x Tecnologia x Año (años del archivo de porcentajes + años
# del Nacional) para detectar también los huecos que no tienen fila en el Excel,
# no solo los que llegan como NaN.
anios_grid = sorted(set(df_porcentajes["Año"]) | set(df_nac_long["Año"]))
regiones = df_porcentajes["Region"].dropna().unique()
tecnologias = df_porcentajes["Tecnologia_Texto_Norm"].dropna().unique()

grid = pd.MultiIndex.from_product(
    [regiones, tecnologias, anios_grid], names=["Region", "Tecnologia_Texto_Norm", "Año"]
).to_frame(index=False)

df_porcentajes = grid.merge(
    df_porcentajes.drop(columns="Tecnologia_Texto"),
    on=["Region", "Tecnologia_Texto_Norm", "Año"],
    how="left",
)

# Orden por año DENTRO de cada Region+Tecnologia: groupby().ffill() resetea el
# arrastre en cada cambio de grupo, así nunca se mezcla una región/tecnología con otra.
df_porcentajes = df_porcentajes.sort_values(["Region", "Tecnologia_Texto_Norm", "Año"])
df_porcentajes["Porcentaje"] = df_porcentajes.groupby(
    ["Region", "Tecnologia_Texto_Norm"]
)["Porcentaje"].ffill()

# Si el primer año de una combinación ya no tiene dato, no hay año anterior del
# cual copiar: ahí sí queda en 0, pero se reporta explícitamente.
sin_arrastre = df_porcentajes["Porcentaje"].isna()
if sin_arrastre.any():
    combos = df_porcentajes.loc[sin_arrastre, ["Region", "Tecnologia_Texto_Norm"]].drop_duplicates()
    print(f"ADVERTENCIA: {len(combos)} combinaciones Region+Tecnologia sin ningún año con porcentaje previo -> quedan en 0:")
    print(combos.to_string(index=False))
    df_porcentajes["Porcentaje"] = df_porcentajes["Porcentaje"].fillna(0)

df_porcentajes.head()

## 4. Cruce (merge) y cálculo del valor regional

In [ ]:
df_regional = pd.merge(
    df_nac_long,
    df_porcentajes,
    on=["Tecnologia_Texto_Norm", "Año"],
    how="left",
)

sin_match = df_regional[df_regional["Region"].isna()]
codigos_con_match = set(df_regional.dropna(subset=["Region"])["Codigo_Tecnologia"])

# Códigos que no cruzaron en NINGÚN año (problema real: revisar el texto en MAPEO_TECNOLOGIA).
# Ya no deberían aparecer años puntuales sueltos: df_porcentajes llega con la grilla
# completa Region x Tecnologia x Año (con relleno hacia adelante), así que un código
# solo queda sin cruce si su texto no existe en absoluto en el archivo de porcentajes.
codigos_totalmente_sin_match = set(sin_match["Codigo_Tecnologia"]) - codigos_con_match
if codigos_totalmente_sin_match:
    print(f"ADVERTENCIA: sin NINGUNA participación regional para: {sorted(codigos_totalmente_sin_match)}")

# Filas sin cruce (tecnología ausente del archivo de porcentajes) se descartan; el
# valor quedará en 0 tras el pivot final (fill_value=0)
df_regional = df_regional.dropna(subset=["Region"]).reset_index(drop=True)

df_regional["Valor_Regional"] = df_regional["Valor_Nacional"] * df_regional["Porcentaje"]

df_regional.head(10)

## 5. Construcción del código TECHNOLOGY regional

In [ ]:
df_regional["Prefijo"] = df_regional["Region"].map(PREFIJO_REGION)

prefijos_faltantes = df_regional[df_regional["Prefijo"].isna()]["Region"].unique()
if len(prefijos_faltantes):
    print(f"ADVERTENCIA: regiones sin prefijo definido: {sorted(prefijos_faltantes)}")

df_regional["TECHNOLOGY"] = df_regional["Prefijo"] + "_" + df_regional["Codigo_Tecnologia"]
df_regional[["Region", "Prefijo", "Codigo_Tecnologia", "TECHNOLOGY", "Año", "Valor_Regional"]].head(10)

## 6. Reestructurar en el formato del SAND Regional (formato ancho)

In [ ]:
df_ancho = df_regional.pivot_table(
    index="TECHNOLOGY", columns="Año", values="Valor_Regional", fill_value=0
)
df_ancho = df_ancho.reindex(columns=[int(c) for c in YEAR_COLS], fill_value=0)
df_ancho = df_ancho.reset_index()
df_ancho.columns = ["TECHNOLOGY"] + [str(c) for c in df_ancho.columns if c != "TECHNOLOGY"]

COLUMNAS_SAND = [
    "Parameter", "REGION", "TECHNOLOGY", "EMISSION", "MODE_OF_OPERATION",
    "FUEL", "TIMESLICE", "STORAGE", "REGION2", "Time indipendent variables",
]

for col in COLUMNAS_SAND:
    if col not in df_ancho.columns:
        df_ancho.insert(0 if col == "Parameter" else len(COLUMNAS_SAND), col, None)

df_ancho["Parameter"] = PARAMETRO
df_ancho["REGION"] = REGION_OSEMOSYS

year_cols_out = [c for c in df_ancho.columns if re.fullmatch(r"\d{4}", str(c))]
df_salida = df_ancho[COLUMNAS_SAND + sorted(year_cols_out, key=int)]

df_salida.head()

## 7. Exportar

In [ ]:
df_salida.to_excel(OUTPUT_PATH, sheet_name="Parameters", index=False)
print(f"Archivo exportado en: {OUTPUT_PATH}")
print(f"Filas exportadas: {len(df_salida)} (esperado: {len(MAPEO_TECNOLOGIA)} códigos x 7 regiones = {len(MAPEO_TECNOLOGIA) * 7})")